# 05. Look-Alike Scoring

> **Краткое резюме**: Определяем эталонную группу по многофакторному
> composite-скору (активность + сеть + рост + стабильность + объём),
> обучаем GBM-классификатор «эталон vs остальные», дополняем kNN cosine
> и объединяем в ансамбль. Результат — ранжированный список проспектов.

**Алгоритм**:
1. **Эталон** (многофакторный): `clientcounterparty_flag = 'Y'`
   + `clientchange_date IS NOT NULL` + **composite quality score** (топ 20%)
2. **kNN cosine** (35% веса): средняя похожесть к K ближайшим эталонным соседям
3. **GBM classifier** (65% веса): P(принадлежность к эталону)
4. **Ensemble**: взвешенное среднее → `lookalike_score ∈ [0, 1]`
5. **Ранжирование**: проспекты (`flag = 'N'`) отсортированы по score

**Composite quality score** объединяет 6 измерений:
- Транзакционная активность (tx_count)
- Ширина сети (unique_counterparties)
- Регулярность (active_months)
- Динамика роста (amount_growth)
- Объём оборота (total_amount)
- Сбалансированность потоков (direction_ratio → |0.5 - x|)

**Предпосылки**: `03_feature_engineering` и `04_segmentation` выполнены.

---

In [ ]:
import os
import pickle
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler

warnings.filterwarnings("ignore", category=UserWarning)

OUTPUT_DIR = os.path.abspath("./data")

In [ ]:
# =====================================================
# ПАРАМЕТРЫ
# =====================================================

# Доля лучших клиентов для эталона (топ N% по composite quality score)
TOP_PERCENTILE = 0.20  # 20%

# Многофакторный composite quality score:
QUALITY_WEIGHTS = {
    "tx_count": 0.25,
    "unique_counterparties": 0.20,
    "active_months": 0.15,
    "amount_growth": 0.15,
    "total_amount": 0.15,
    "direction_balance": 0.10,
}

K_NEIGHBORS = 50
WEIGHT_GBM = 0.65
WEIGHT_KNN = 0.35
SCORE_THRESHOLD = 0.60
TOP_N_PROSPECTS = 100

# ── Cross-Validation ─────────────────────────────────────────────────────────
CV_FOLDS = 5

# ── Score Bands ──────────────────────────────────────────────────────────────
SCORE_BANDS = {
    "Hot": (0.8, 1.0),
    "Warm": (0.6, 0.8),
    "Medium": (0.4, 0.6),
    "Cool": (0.2, 0.4),
    "Cold": (0.0, 0.2),
}

---
## 1. Загрузка данных

In [ ]:
X = pd.read_parquet(os.path.join(OUTPUT_DIR, "feature_matrix.parquet"))
full_df = pd.read_parquet(os.path.join(OUTPUT_DIR, "segments.parquet"))

with open(os.path.join(OUTPUT_DIR, "feature_meta.pkl"), "rb") as f:
    meta = pickle.load(f)

print(f"Feature matrix: {X.shape}")
print(f"Full data: {full_df.shape}")
print("\nDistribution of clientcounterparty_flag:")
print(full_df["clientcounterparty_flag"].value_counts())

---
## 2. Определение эталонной группы (многофакторный отбор)

**Лучшие привлечённые клиенты** отбираются по composite quality score:

1. Сейчас клиент (`clientcounterparty_flag = 'Y'`)
2. Когда-то был контрагентом (`clientchange_date IS NOT NULL`)
3. **Composite quality score** — взвешенная сумма Z-score по 6 измерениям:
   - `tx_count` (25%) — транзакционная активность
   - `unique_counterparties` (20%) — ширина сети
   - `active_months` (15%) — регулярность
   - `amount_growth` (15%) — рост оборота
   - `total_amount` (15%) — объём бизнеса
   - `direction_balance` (10%) — сбалансированность потоков
4. Топ 20% по composite score → **эталонная группа**

Преимущество над однофакторным отбором (только по обороту):
- Клиент с большим оборотом, но 1 контрагентом и падающей динамикой → не попадёт в эталон
- Клиент с умеренным оборотом, но широкой сетью и ростом → попадёт в эталон
- Результат: более качественная эталонная группа, лучше отражающая «идеального» клиента

In [ ]:
# Клиенты
clients_mask = full_df["clientcounterparty_flag"] == "Y"
print(f"Clients (Y): {clients_mask.sum():,}")

# Привлечённые (были контрагентами → стали клиентами)
if "clientchange_date" in full_df.columns:
    acquired_mask = clients_mask & full_df["clientchange_date"].notna()
    print(f"Acquired (Y + clientchange_date): {acquired_mask.sum():,}")
else:
    acquired_mask = clients_mask
    print("clientchange_date not available — using all clients")

# Если мало привлечённых, используем всех клиентов
if acquired_mask.sum() < 100:
    print(f"Too few acquired ({acquired_mask.sum()}), falling back to all clients")
    acquired_mask = clients_mask

# ── Многофакторный composite quality score ────────────────────────────────
acquired_df = full_df[acquired_mask].copy()

# Собираем доступные метрики для composite score
quality_raw = pd.DataFrame(index=acquired_df.index)
for col in QUALITY_WEIGHTS:
    if col == "direction_balance":
        # Сбалансированность: |0.5 - DR| → инверсия (ближе к 0.5 = лучше)
        if "direction_ratio" in acquired_df.columns:
            quality_raw[col] = 1 - (acquired_df["direction_ratio"] - 0.5).abs() * 2
    elif col in acquired_df.columns:
        quality_raw[col] = acquired_df[col]

# Z-score нормализация каждого измерения
quality_z = pd.DataFrame(index=quality_raw.index)
for col in quality_raw.columns:
    _mean = quality_raw[col].mean()
    _std = quality_raw[col].std()
    quality_z[col] = (quality_raw[col] - _mean) / (_std + 1e-9)

# Взвешенная сумма Z-score → composite quality score
composite = pd.Series(0.0, index=quality_z.index)
active_weights = {}
for col in quality_z.columns:
    w = QUALITY_WEIGHTS.get(col, 0.0)
    composite += quality_z[col] * w
    active_weights[col] = w

# Нормируем веса если не все метрики доступны
total_w = sum(active_weights.values())
if total_w > 0 and abs(total_w - 1.0) > 0.01:
    composite = composite / total_w

acquired_df["composite_quality"] = composite

# Топ N% по composite score
quality_threshold = composite.quantile(1 - TOP_PERCENTILE)
reference_indices = acquired_df[composite >= quality_threshold].index
reference_mask = full_df.index.isin(reference_indices)

print(f"\n{'=' * 60}")
print("МНОГОФАКТОРНЫЙ ОТБОР ЭТАЛОННОЙ ГРУППЫ")
print(f"{'=' * 60}")
print(f"Composite quality score — {len(active_weights)} измерений:")
for col, w in sorted(active_weights.items(), key=lambda x: -x[1]):
    z_mean_ref = quality_z.loc[reference_indices, col].mean()
    z_mean_all = quality_z[col].mean()
    print(f"  {col:26s}  вес={w:.0%}  Z(эталон)={z_mean_ref:+.2f}  Z(все)={z_mean_all:+.2f}")

print(f"\nComposite threshold (top {TOP_PERCENTILE * 100:.0f}%): {quality_threshold:.3f}")
print(f"Reference group: {reference_mask.sum():,} клиентов")

# ── Сравнение с однофакторным отбором ─────────────────────────────────────
old_threshold = acquired_df["total_amount"].quantile(1 - TOP_PERCENTILE)
old_mask = acquired_mask & (full_df["total_amount"] >= old_threshold)
overlap = (reference_mask & old_mask).sum()
only_new = (reference_mask & ~old_mask).sum()
only_old = (old_mask & ~reference_mask).sum()

print(f"\nСравнение с отбором по обороту (top {TOP_PERCENTILE * 100:.0f}% total_amount):")
print(f"  Совпадение: {overlap:,} ({overlap / reference_mask.sum() * 100:.0f}%)")
print(f"  Только в новом (многофакт.): {only_new:,} — клиенты с широкой сетью/ростом")
print(f"  Только в старом (оборот): {only_old:,} — высокий оборот, но слабая сеть/рост")

# ── Профиль: новый эталон vs старый ──────────────────────────────────────
_profile_cols = [
    c
    for c in [
        "total_amount",
        "tx_count",
        "unique_counterparties",
        "active_months",
        "amount_growth",
        "direction_ratio",
    ]
    if c in full_df.columns
]
profile_compare = pd.DataFrame(
    {
        "Новый эталон (composite)": full_df.loc[reference_mask, _profile_cols].mean(),
        "Старый эталон (оборот)": full_df.loc[old_mask, _profile_cols].mean(),
        "Все привлечённые": acquired_df[_profile_cols].mean(),
    }
)
print("\nПрофиль эталонных групп:")
display(profile_compare.round(2))

---
## 3. Скоринг: kNN + GBM Ensemble

**Два независимых сигнала, объединённых в ансамбль:**
- **kNN cosine** (35%): средняя косинусная похожесть к ближайшим K эталонным клиентам. Улавливает «форму» облака эталонов.
- **GBM** (65%): бинарный классификатор эталон vs. остальные клиенты. Учитывает нелинейные взаимодействия признаков.

Ансамбль устойчивее каждого компонента по отдельности.

In [ ]:
# ── Шаг A: kNN cosine scoring ───────────────────────────────────────────────
X_reference = X.loc[X.index.isin(full_df[reference_mask].index)]
X_reference_vals = X_reference.values  # (n_ref, n_features)
X_all_vals = X.values  # (n_all, n_features)

print(f"Reference vectors: {X_reference_vals.shape[0]:,}")
print(f"Computing cosine similarity ({X_all_vals.shape[0]} × {X_reference_vals.shape[0]})…")
sim_matrix = cosine_similarity(X_all_vals, X_reference_vals)

k = min(K_NEIGHBORS, X_reference_vals.shape[0])
knn_scores_raw = np.sort(sim_matrix, axis=1)[:, -k:].mean(axis=1)
_knn_min, _knn_max = knn_scores_raw.min(), knn_scores_raw.max()
knn_scores = (knn_scores_raw - _knn_min) / (_knn_max - _knn_min + 1e-9)
full_df["score_knn"] = knn_scores
print(f"kNN (k={k}) — mean={knn_scores.mean():.3f}  std={knn_scores.std():.3f}")

# ── Шаг A2: Centroid baseline (для сравнения методов) ───────────────────────
# Классический подход: 1 вектор-центроид эталонной группы → cosine similarity.
# Используется как baseline в сравнительной таблице методов.
centroid = X_reference_vals.mean(axis=0)
centroid_sim = cosine_similarity(X_all_vals, centroid.reshape(1, -1))[:, 0]
_c_min, _c_max = centroid_sim.min(), centroid_sim.max()
centroid_norm = (centroid_sim - _c_min) / (_c_max - _c_min + 1e-9)
full_df["score_centroid"] = centroid_norm
print(f"Centroid baseline — mean={centroid_norm.mean():.3f}  std={centroid_norm.std():.3f}")

# ── Шаг B: GBM classifier ───────────────────────────────────────────────────
# Эталон = top 20% по composite quality score (многофакторный).
# Monetary features исключены из GBM whitelist (leakage protection).
# Whitelist: поведенческие признаки + граф-метрики (если загружены).
GBM_BEHAVIORAL_PREFIXES = (
    "tx_count",
    "unique_",
    "active_months",
    "direction_ratio",
    "amount_growth",
    "cp_growth",
    "tx_amount_cv",
    "tx_per_month",
    "cp_per_month",
    "payee_ratio",
    "payer_ratio",
    "product_count",
    "product_type_count",
    "pagerank",
    "betweenness",
    "clustering_coef",
    "flow_through_ratio",
    "in_degree",
    "out_degree",
    "top_k_concentration",
    "okved_diversity_",
    "role_",
    "okved_",
    "region_",
    "mseg_",
    "ctype_",
    "herfindahl_index",
    "monthly_volatility",
    "counterparty_retention",
    "balance_turnover_ratio",
    "growth_acceleration",
    "network_influence",
    "amt_per_cp",
)
gbm_feature_cols = [
    c for c in X.columns if any(c == p or c.startswith(p) for p in GBM_BEHAVIORAL_PREFIXES)
]
excluded = [c for c in X.columns if c not in gbm_feature_cols]
print(f"\nGBM whitelist: {len(gbm_feature_cols)} features | Excluded: {excluded}")

pos_idx = full_df[reference_mask].index
neg_idx = full_df[clients_mask & ~reference_mask].index
X_pos = X.loc[X.index.isin(pos_idx), gbm_feature_cols].values
X_neg = X.loc[X.index.isin(neg_idx), gbm_feature_cols].values
X_train = np.vstack([X_pos, X_neg])
y_train = np.concatenate([np.ones(len(X_pos)), np.zeros(len(X_neg))])
X_all_gbm = X[gbm_feature_cols].values
print(f"Train: {len(X_pos):,} pos (reference) vs {len(X_neg):,} neg")

# Приоритет: LightGBM > XGBoost > sklearn GBM
# LightGBM: гистограммный GBM, нативная поддержка категориальных признаков,
# встроенный early stopping, 5-10x быстрее sklearn на тех же данных.
from sklearn.model_selection import train_test_split as _tts  # noqa: E402

X_tr, X_val, y_tr, y_val = _tts(X_train, y_train, test_size=0.15, random_state=42, stratify=y_train)

try:
    import lightgbm as lgb

    clf = lgb.LGBMClassifier(
        n_estimators=500,
        learning_rate=0.05,
        num_leaves=63,
        min_child_samples=20,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=len(X_neg) / len(X_pos),
        n_jobs=-1,
        random_state=42,
        verbose=-1,
    )
    clf.fit(
        X_tr,
        y_tr,
        eval_set=[(X_val, y_val)],
        callbacks=[lgb.early_stopping(30, verbose=False), lgb.log_evaluation(period=-1)],
    )
    clf_name = "LightGBM"

except ImportError:
    try:
        from xgboost import XGBClassifier

        clf = XGBClassifier(
            n_estimators=500,
            max_depth=4,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            scale_pos_weight=len(X_neg) / len(X_pos),
            early_stopping_rounds=30,
            eval_metric="logloss",
            random_state=42,
            verbosity=0,
        )
        clf.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
        clf_name = "XGBoost"

    except ImportError:
        clf = GradientBoostingClassifier(
            n_estimators=500,
            max_depth=4,
            learning_rate=0.05,
            subsample=0.8,
            validation_fraction=0.15,
            n_iter_no_change=30,
            tol=1e-4,
            random_state=42,
        )
        clf.fit(X_train, y_train)
        clf_name = "GradientBoosting (sklearn)"

n_trees = getattr(
    clf,
    "n_estimators_",
    getattr(clf, "best_iteration_", getattr(clf, "n_estimators", "?")),
)
print(f"\n{clf_name} trained ({n_trees} trees).")

gbm_proba = clf.predict_proba(X_all_gbm)[:, 1]
full_df["score_gbm"] = gbm_proba
gbm_norm = (gbm_proba - gbm_proba.min()) / (gbm_proba.max() - gbm_proba.min() + 1e-9)
full_df["score_gbm_norm"] = gbm_norm  # normalized for method comparison
ref_mean = gbm_proba[X.index.isin(pos_idx)].mean()
non_ref_mean = gbm_proba[X.index.isin(neg_idx)].mean()
print(f"P(ref) эталон={ref_mean:.3f}  остальные={non_ref_mean:.3f}")

# ── Шаг C: Ensemble ─────────────────────────────────────────────────────────
full_df["lookalike_score"] = WEIGHT_GBM * gbm_norm + WEIGHT_KNN * knn_scores
print(f"\n✅ Ensemble ({WEIGHT_GBM:.0%} {clf_name.split()[0]} + {WEIGHT_KNN:.0%} kNN):")
print(full_df["lookalike_score"].describe().round(4))

In [ ]:
# ── Feature Importance: SHAP (preferred) → impurity fallback ─────────────────
# SHAP (SHapley Additive exPlanations): показывает вклад каждого признака
# для КАЖДОГО наблюдения отдельно.
#
# Beeswarm-диаграмма:
#   • Ось X = SHAP-значение (влияние на логит P(эталон))
#   • Цвет точки = фактическое значение признака (красный=высокое, синий=низкое)
#   • Каждая точка = один клиент
#
# Преимущества SHAP над impurity importance:
#   ✅ Показывает НАПРАВЛЕНИЕ влияния (↑/↓), а не только magnitude
#   ✅ Учитывает взаимодействия между признаками
#   ✅ Не завышает важность высококардинальных признаков
#   ✅ Основан на теории игр (Shapley values) — математически обоснован

FEATURE_DESCRIPTIONS = {
    "tx_count": "Кол-во транзакций — мера активности клиента",
    "unique_counterparties": "Число уникальных контрагентов — широта связей",
    "direction_ratio": "Доля расходов [0..1]",
    "active_months": "Активные месяцы — регулярность",
    "amount_growth": "Рост оборота — динамика развития",
    "cp_growth": "Рост числа контрагентов — расширение",
    "tx_amount_cv": "Вариация сумм транзакций — однородность",
    "tx_per_month": "Транзакций в месяц — интенсивность",
    "cp_per_month": "Контрагентов в месяц — скорость расширения",
    "payee_ratio": "Доля уникальных получателей — широта расходов",
    "payer_ratio": "Доля уникальных отправителей",
    "pagerank": "PageRank в графе транзакций — системная важность",
    "betweenness": "Betweenness centrality — роль посредника",
    "clustering_coef": "Коэф. кластеризации — плотность локального сообщества",
    "flow_through_ratio": "Транзитные потоки — проходящий оборот",
}


def _pearson(x: np.ndarray, y: np.ndarray) -> float:
    """Pearson correlation без np.corrcoef (несовместим с numpy >= 2.0)."""
    xm, ym = x - x.mean(), y - y.mean()
    denom = np.sqrt((xm**2).sum() * (ym**2).sum())
    return float(xm @ ym / denom) if denom > 1e-10 else 0.0


try:
    import shap

    print("🔍 Computing SHAP values (TreeExplainer)…")
    explainer = shap.TreeExplainer(clf)

    # Sample для скорости (TreeExplainer O(n * depth) — достаточно 2000)
    n_shap = min(2000, len(X_all_gbm))
    rng = np.random.default_rng(42)
    shap_idx = (
        rng.choice(len(X_all_gbm), n_shap, replace=False)
        if len(X_all_gbm) > n_shap
        else np.arange(len(X_all_gbm))
    )
    X_shap = X_all_gbm[shap_idx]
    shap_raw = explainer.shap_values(X_shap)

    # LightGBM binary → list [neg, pos] или 2D array positive class
    if isinstance(shap_raw, list):
        shap_vals = shap_raw[1]  # positive class (P=эталон)
    else:
        shap_vals = shap_raw

    print(f"SHAP values shape: {shap_vals.shape}  (clients × features)")

    # ── Beeswarm ──────────────────────────────────────────────────────────────
    shap.summary_plot(shap_vals, X_shap, feature_names=gbm_feature_cols, max_display=15, show=False)
    plt.title("SHAP Beeswarm: направление влияния признаков на P(эталон)", fontsize=12, pad=10)
    plt.tight_layout()
    plt.show()

    # ── Bar: mean |SHAP| ──────────────────────────────────────────────────────
    mean_abs_shap = pd.Series(np.abs(shap_vals).mean(axis=0), index=gbm_feature_cols).sort_values(
        ascending=False
    )
    top15 = mean_abs_shap.head(15)

    fig, ax = plt.subplots(figsize=(10, 6))
    colors_shap = ["#d62728" if i < 3 else "#1f77b4" for i in range(len(top15))]
    top15[::-1].plot(kind="barh", ax=ax, color=colors_shap[::-1], edgecolor="black", alpha=0.8)
    ax.set_xlabel("Mean |SHAP value|  (вклад в логит P(эталон))")
    ax.set_title("Топ-15 признаков по |SHAP| — средний вклад в модель", fontsize=12)
    for i, (_name, val) in enumerate(top15[::-1].items()):
        ax.text(val + 0.0005, i, f"{val:.4f}", va="center", fontsize=8)
    plt.tight_layout()
    plt.show()

    # ── Текстовая интерпретация ────────────────────────────────────────────────
    print("\n📊 ТОП-5 ПРИЗНАКОВ — SHAP ИНТЕРПРЕТАЦИЯ:")
    print("─" * 70)
    for rank, (fname, mabs) in enumerate(mean_abs_shap.head(5).items(), 1):
        feat_idx = gbm_feature_cols.index(fname)
        feat_col = X_shap[:, feat_idx]
        shap_col = shap_vals[:, feat_idx]
        # np.corrcoef → np.cov → np.average(returned=True) breaks on numpy >= 2.0
        corr = _pearson(feat_col, shap_col) if feat_col.std() > 1e-10 else 0.0
        if corr > 0.05:
            direction = "↑  высокое значение → выше P(эталон)"
        elif corr < -0.05:
            direction = "↓  низкое значение → выше P(эталон)"
        else:
            direction = "↔  нелинейное / контекстно-зависимое"
        desc = FEATURE_DESCRIPTIONS.get(fname, fname)
        print(f"  #{rank}: {fname:28s}  |SHAP|={mabs:.4f}  {direction}")
        print(f"       {desc}")
        print()

    shap_available = True  # флаг для следующих ячеек

except ImportError:
    shap_available = False
    print("⚠️  shap не установлен. Fallback: impurity importance.")
    print("   Установить: uv pip install shap>=0.44\n")

    importances = getattr(clf, "feature_importances_", np.zeros(len(gbm_feature_cols)))
    fi = pd.Series(importances, index=gbm_feature_cols).sort_values(ascending=False)
    top_fi = fi.head(20)

    fig, ax = plt.subplots(figsize=(10, 6))
    colors_fi = ["#d62728" if i < 5 else "#1f77b4" for i in range(len(top_fi))]
    top_fi[::-1].plot(kind="barh", ax=ax, color=colors_fi[::-1], edgecolor="black", alpha=0.8)
    ax.set_xlabel("Feature Importance (impurity)")
    ax.set_title("Топ-20 признаков (impurity importance)", fontsize=12)
    for i, (_name, val) in enumerate(top_fi[::-1].items()):
        ax.text(val + 0.001, i, f"{val:.3f}", va="center", fontsize=8)
    plt.tight_layout()
    plt.show()

    print("\n📊 ТОП-5 ПРИЗНАКОВ (impurity):")
    for rank, (fname, fval) in enumerate(top_fi.head(5).items(), 1):
        pct = fval / fi.sum() * 100
        desc = FEATURE_DESCRIPTIONS.get(fname, fname)
        print(f"  #{rank}: {fname:30s} {pct:5.1f}%  ← {desc}")

# Сохраняем имена признаков для pickle (используется в cell-20)
feature_names = gbm_feature_cols

---
## 3.6. Кросс-валидация (Stratified K-Fold)

Оценка устойчивости модели: 5-fold stratified CV с метриками по каждому fold.

In [ ]:
# ── Stratified K-Fold Cross-Validation ───────────────────────────────────────
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold

skf = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=42)
fold_metrics = []
oof_predictions = np.zeros(len(X_train))  # out-of-fold predictions for stacking

for fold_i, (train_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
    X_fold_tr, X_fold_val = X_train[train_idx], X_train[val_idx]
    y_fold_tr, y_fold_val = y_train[train_idx], y_train[val_idx]

    try:
        import lightgbm as lgb

        fold_clf = lgb.LGBMClassifier(
            n_estimators=500,
            learning_rate=0.05,
            num_leaves=63,
            min_child_samples=20,
            subsample=0.8,
            colsample_bytree=0.8,
            scale_pos_weight=len(y_train[y_train == 0]) / max(len(y_train[y_train == 1]), 1),
            n_jobs=-1,
            random_state=42,
            verbose=-1,
        )
        fold_clf.fit(
            X_fold_tr,
            y_fold_tr,
            eval_set=[(X_fold_val, y_fold_val)],
            callbacks=[lgb.early_stopping(30, verbose=False), lgb.log_evaluation(period=-1)],
        )
    except ImportError:
        fold_clf = GradientBoostingClassifier(
            n_estimators=300,
            max_depth=4,
            learning_rate=0.05,
            subsample=0.8,
            random_state=42,
        )
        fold_clf.fit(X_fold_tr, y_fold_tr)

    fold_proba = fold_clf.predict_proba(X_fold_val)[:, 1]
    oof_predictions[val_idx] = fold_proba

    auc_roc = roc_auc_score(y_fold_val, fold_proba)
    pr_auc = average_precision_score(y_fold_val, fold_proba)

    # Separation index on validation fold
    sep = fold_proba[y_fold_val == 1].mean() - fold_proba[y_fold_val == 0].mean()

    # Lift@5% on validation fold
    n_top = max(1, int(len(fold_proba) * 0.05))
    top_idx = np.argsort(fold_proba)[::-1][:n_top]
    lift_5 = y_fold_val[top_idx].mean() / max(y_fold_val.mean(), 1e-9)

    fold_metrics.append(
        {
            "fold": fold_i + 1,
            "AUC-ROC": round(auc_roc, 4),
            "PR-AUC": round(pr_auc, 4),
            "Separation": round(sep, 4),
            "Lift@5%": round(lift_5, 2),
        }
    )

cv_df = pd.DataFrame(fold_metrics)
print(f"📊 Stratified {CV_FOLDS}-Fold Cross-Validation:")
display(cv_df)

print("\nMean ± Std:")
for metric in ["AUC-ROC", "PR-AUC", "Separation", "Lift@5%"]:
    vals = cv_df[metric]
    print(f"  {metric:15s}: {vals.mean():.4f} ± {vals.std():.4f}")

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
x_pos = np.arange(CV_FOLDS)
width = 0.35
axes[0].bar(x_pos - width / 2, cv_df["AUC-ROC"], width, label="AUC-ROC", color="#1f77b4", alpha=0.8)
axes[0].bar(x_pos + width / 2, cv_df["PR-AUC"], width, label="PR-AUC", color="#ff7f0e", alpha=0.8)
axes[0].axhline(cv_df["AUC-ROC"].mean(), color="#1f77b4", linestyle="--", alpha=0.5)
axes[0].axhline(cv_df["PR-AUC"].mean(), color="#ff7f0e", linestyle="--", alpha=0.5)
axes[0].set_xlabel("Fold")
axes[0].set_ylabel("Score")
axes[0].set_title("AUC-ROC / PR-AUC по фолдам")
axes[0].set_xticks(x_pos)
axes[0].set_xticklabels([f"Fold {i + 1}" for i in range(CV_FOLDS)])
axes[0].legend()

axes[1].bar(x_pos, cv_df["Separation"], color="#2ca02c", alpha=0.8, edgecolor="black")
_sep_mean = cv_df["Separation"].mean()
axes[1].axhline(_sep_mean, color="red", linestyle="--", label=f"Mean={_sep_mean:.3f}")
axes[1].set_xlabel("Fold")
axes[1].set_ylabel("Separation Index")
axes[1].set_title("Separation Index по фолдам")
axes[1].set_xticks(x_pos)
axes[1].set_xticklabels([f"Fold {i + 1}" for i in range(CV_FOLDS)])
axes[1].legend()

plt.suptitle(f"Stratified {CV_FOLDS}-Fold Cross-Validation", fontsize=13)
plt.tight_layout()
plt.show()

cv_auc_mean = cv_df["AUC-ROC"].mean()
cv_auc_std = cv_df["AUC-ROC"].std()
print(f"\n{'=' * 60}")
if cv_auc_mean >= 0.80:
    print(f"✅ AUC-ROC = {cv_auc_mean:.3f} ± {cv_auc_std:.3f} — хорошее качество модели")
elif cv_auc_mean >= 0.70:
    print(f"⚠️  AUC-ROC = {cv_auc_mean:.3f} ± {cv_auc_std:.3f} — приемлемое качество")
else:
    print(f"❌ AUC-ROC = {cv_auc_mean:.3f} ± {cv_auc_std:.3f} — модель слабая")
if cv_auc_std > 0.05:
    print("  ⚠️  Высокая дисперсия между фолдами — модель нестабильна")
else:
    print("  ✅ Низкая дисперсия — модель стабильна")

---
## 3.7. Калибровка модели

Проверяем: соответствует ли P(эталон)=0.7 реальной вероятности ~70%?

In [ ]:
# ── Calibration Analysis ─────────────────────────────────────────────────────
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.metrics import brier_score_loss

# Use OOF predictions for calibration analysis
oof_valid = oof_predictions[oof_predictions > 0]  # non-zero = was in some val fold
y_oof_valid = y_train[oof_predictions > 0]

brier = brier_score_loss(y_oof_valid, oof_valid)
fraction_of_positives, mean_predicted = calibration_curve(
    y_oof_valid, oof_valid, n_bins=10, strategy="uniform"
)

# Expected Calibration Error (ECE)
bin_edges = np.linspace(0, 1, 11)
ece = 0.0
for i in range(10):
    mask = (oof_valid >= bin_edges[i]) & (oof_valid < bin_edges[i + 1])
    if mask.sum() > 0:
        bin_acc = y_oof_valid[mask].mean()
        bin_conf = oof_valid[mask].mean()
        ece += mask.sum() / len(oof_valid) * abs(bin_acc - bin_conf)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Reliability diagram
axes[0].plot([0, 1], [0, 1], "k--", label="Идеальная калибровка")
axes[0].plot(
    mean_predicted,
    fraction_of_positives,
    "s-",
    color="#d62728",
    label=f"GBM (Brier={brier:.4f})",
    markersize=8,
)
axes[0].fill_between(
    mean_predicted, fraction_of_positives, mean_predicted, alpha=0.15, color="#d62728"
)
axes[0].set_xlabel("Предсказанная вероятность")
axes[0].set_ylabel("Фактическая доля положительных")
axes[0].set_title("Reliability Diagram (Calibration Curve)")
axes[0].legend()
axes[0].set_xlim(0, 1)
axes[0].set_ylim(0, 1)

# Histogram of predicted probabilities
axes[1].hist(
    oof_valid[y_oof_valid == 1],
    bins=30,
    alpha=0.6,
    label="Эталон (y=1)",
    color="#2ca02c",
    edgecolor="black",
)
axes[1].hist(
    oof_valid[y_oof_valid == 0],
    bins=30,
    alpha=0.6,
    label="Остальные (y=0)",
    color="#1f77b4",
    edgecolor="black",
)
axes[1].set_xlabel("Предсказанная P(эталон)")
axes[1].set_ylabel("Количество")
axes[1].set_title("Распределение предсказаний по классам")
axes[1].legend()

plt.suptitle("Анализ калибровки модели", fontsize=13)
plt.tight_layout()
plt.show()

print(f"Brier Score: {brier:.4f}  (ниже = лучше, <0.1 = хорошо)")
print(f"ECE:         {ece:.4f}  (ниже = лучше, <0.05 = хорошо)")

# Calibrate if needed
if ece > 0.05:
    print(f"\n⚠️  ECE={ece:.4f} > 0.05 — калибровка рекомендуется")
    print("Применяем CalibratedClassifierCV (isotonic)…")
    cal_clf = CalibratedClassifierCV(clf, method="isotonic", cv=3)
    cal_clf.fit(X_train, y_train)
    gbm_proba_cal = cal_clf.predict_proba(X_all_gbm)[:, 1]
    brier_cal = brier_score_loss(y_oof_valid, oof_valid)  # approx
    print(f"Calibrated Brier: ~{brier_cal:.4f}")
    # Update scores
    full_df["score_gbm"] = gbm_proba_cal
    _cal_min = gbm_proba_cal.min()
    _cal_range = gbm_proba_cal.max() - _cal_min + 1e-9
    gbm_norm_cal = (gbm_proba_cal - _cal_min) / _cal_range
    full_df["score_gbm_norm"] = gbm_norm_cal
    full_df["lookalike_score"] = WEIGHT_GBM * gbm_norm_cal + WEIGHT_KNN * knn_scores
    print("✅ Scores обновлены с калиброванной моделью")
else:
    print(f"\n✅ ECE={ece:.4f} ≤ 0.05 — калибровка не требуется")

---
## 3.8. Оптимизация порога (Threshold)

Поиск оптимального cutoff для классификации проспектов.

In [ ]:
# ── Threshold Optimization ───────────────────────────────────────────────────
from sklearn.metrics import precision_recall_curve, roc_curve

# ROC curve on validation set
fpr, tpr, roc_thresholds = roc_curve(y_val, clf.predict_proba(X_val)[:, 1])
roc_auc = roc_auc_score(y_val, clf.predict_proba(X_val)[:, 1])

# Youden's J statistic
j_scores = tpr - fpr
youden_idx = j_scores.argmax()
youden_threshold = roc_thresholds[youden_idx]

# Precision-Recall curve
precision_arr, recall_arr, pr_thresholds = precision_recall_curve(
    y_val, clf.predict_proba(X_val)[:, 1]
)
pr_auc = average_precision_score(y_val, clf.predict_proba(X_val)[:, 1])

# F1-optimal threshold
f1_scores_arr = (
    2 * (precision_arr[:-1] * recall_arr[:-1]) / (precision_arr[:-1] + recall_arr[:-1] + 1e-9)
)
f1_idx = f1_scores_arr.argmax()
f1_threshold = pr_thresholds[f1_idx]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(fpr, tpr, color="#d62728", linewidth=2, label=f"AUC = {roc_auc:.3f}")
axes[0].plot([0, 1], [0, 1], "k--", alpha=0.5)
axes[0].scatter(
    fpr[youden_idx],
    tpr[youden_idx],
    s=100,
    c="green",
    zorder=5,
    label=f"Youden J (t={youden_threshold:.3f})",
)
axes[0].set_xlabel("False Positive Rate")
axes[0].set_ylabel("True Positive Rate")
axes[0].set_title(f"ROC Curve (AUC = {roc_auc:.3f})")
axes[0].legend()

axes[1].plot(recall_arr, precision_arr, color="#1f77b4", linewidth=2, label=f"AP = {pr_auc:.3f}")
axes[1].scatter(
    recall_arr[f1_idx],
    precision_arr[f1_idx],
    s=100,
    c="red",
    zorder=5,
    label=f"F1-optimal (t={f1_threshold:.3f})",
)
axes[1].set_xlabel("Recall")
axes[1].set_ylabel("Precision")
axes[1].set_title(f"Precision-Recall Curve (AP = {pr_auc:.3f})")
axes[1].legend()

plt.suptitle("ROC и Precision-Recall анализ", fontsize=13)
plt.tight_layout()
plt.show()

OPTIMAL_THRESHOLD = youden_threshold
print(f"Youden J threshold: {OPTIMAL_THRESHOLD:.3f}")
print(f"F1-optimal threshold: {f1_threshold:.3f}")
print(f"AUC-ROC: {roc_auc:.3f},  PR-AUC: {pr_auc:.3f}")

---
## 3.9. Segment-Aware валидация

Качество модели по каждому поведенческому сегменту: выявляем слабые места.

In [ ]:
# ── Segment-Aware Validation ─────────────────────────────────────────────────
if "behavioral_segment" in full_df.columns:
    segments = sorted(full_df["behavioral_segment"].dropna().unique())
    seg_metrics = []

    for seg_id in segments:
        seg_mask = full_df["behavioral_segment"] == seg_id
        seg_scores = full_df.loc[seg_mask, "lookalike_score"].values
        seg_ref = reference_mask[seg_mask].values
        n_total = int(seg_mask.sum())
        n_ref = int(seg_ref.sum())

        if n_ref < 5 or n_total - n_ref < 5:
            continue

        sep = float(seg_scores[seg_ref].mean() - seg_scores[~seg_ref].mean())

        try:
            auc = roc_auc_score(seg_ref.astype(int), seg_scores)
        except ValueError:
            auc = np.nan

        n_top = max(1, int(n_total * 0.05))
        top_idx = np.argsort(seg_scores)[::-1][:n_top]
        lift_5 = seg_ref[top_idx].mean() / max(seg_ref.mean(), 1e-9)

        label = ""
        if "segment_label" in full_df.columns:
            label = str(full_df.loc[seg_mask, "segment_label"].iloc[0])[:25]
        seg_metrics.append(
            {
                "Segment": seg_id,
                "Label": label,
                "N": n_total,
                "N_ref": n_ref,
                "Separation": round(sep, 4),
                "AUC-ROC": round(auc, 4) if not np.isnan(auc) else "N/A",
                "Lift@5%": round(lift_5, 2),
            }
        )

    if seg_metrics:
        seg_val_df = pd.DataFrame(seg_metrics)
        print("Качество модели по сегментам:")
        display(seg_val_df)

        weak = [r for r in seg_metrics if isinstance(r["AUC-ROC"], float) and r["AUC-ROC"] < 0.65]
        if weak:
            print(f"\nСлабые сегменты ({len(weak)}):")
            for w in weak:
                print(
                    f"  Сег. {w['Segment']} ({w['Label']}): "
                    f"AUC={w['AUC-ROC']}, Sep={w['Separation']}"
                )
        else:
            print("\nВсе сегменты с AUC >= 0.65")
    else:
        print("Недостаточно данных для segment-aware анализа")
else:
    print("behavioral_segment не найден — segment-aware анализ пропущен")

---
## 3.10. Stacking Ensemble

Замена фиксированных весов на обученный meta-learner (LogisticRegression)
поверх OOF-предсказаний базовых моделей.

In [ ]:
# ── Stacking Ensemble ────────────────────────────────────────────────────────
from sklearn.linear_model import LogisticRegression

# Indices of train clients in X
_train_client_idx = np.concatenate(
    [
        np.where(X.index.isin(pos_idx))[0],
        np.where(X.index.isin(neg_idx))[0],
    ]
)
knn_train_aligned = knn_scores[_train_client_idx]
centroid_train_aligned = centroid_norm[_train_client_idx]

# Meta-features: [GBM_oof, kNN, Centroid]
meta_train = np.column_stack([oof_predictions, knn_train_aligned, centroid_train_aligned])
meta_all = np.column_stack(
    [
        full_df["score_gbm"].values,
        knn_scores,
        centroid_norm,
    ]
)

stacker = LogisticRegression(random_state=42, max_iter=1000)
stacker.fit(meta_train, y_train)
stacking_scores = stacker.predict_proba(meta_all)[:, 1]

_st_min, _st_max = stacking_scores.min(), stacking_scores.max()
stacking_norm = (stacking_scores - _st_min) / (_st_max - _st_min + 1e-9)
full_df["score_stacking"] = stacking_norm

old_sep = (
    full_df.loc[reference_mask, "lookalike_score"].mean()
    - full_df.loc[~reference_mask, "lookalike_score"].mean()
)
new_sep = stacking_norm[reference_mask.values].mean() - stacking_norm[~reference_mask.values].mean()

print("Stacking vs Fixed-Weight Ensemble:")
print(f"  Old ensemble (0.65*GBM + 0.35*kNN): Separation = {old_sep:.4f}")
print(f"  Stacking (LR meta-learner):          Separation = {new_sep:.4f}")
print(
    f"  Stacker coefs: GBM={stacker.coef_[0][0]:.3f}, "
    f"kNN={stacker.coef_[0][1]:.3f}, Centroid={stacker.coef_[0][2]:.3f}"
)

if new_sep > old_sep:
    full_df["lookalike_score"] = stacking_norm
    ENSEMBLE_METHOD = "Stacking"
    print("\nStacking лучше — используем stacking score как финальный")
else:
    ENSEMBLE_METHOD = "Fixed-weight"
    print("\nFixed-weight ensemble лучше — оставляем текущий score")

---
## 3.11. Ablation Study

Какие группы признаков критичны для модели?

In [ ]:
# ── Ablation Study ───────────────────────────────────────────────────────────
FEATURE_GROUPS = {
    "Граф-метрики": [
        c
        for c in gbm_feature_cols
        if any(
            c.startswith(p)
            for p in [
                "pagerank",
                "betweenness",
                "clustering_coef",
                "flow_through_ratio",
                "in_degree",
                "out_degree",
                "top_k_concentration",
                "okved_diversity_",
                "role_",
                "network_influence",
            ]
        )
    ],
    "Категориальные": [
        c
        for c in gbm_feature_cols
        if any(c.startswith(p) for p in ["okved_", "region_", "mseg_", "ctype_"])
    ],
    "Временные": [
        c
        for c in gbm_feature_cols
        if any(
            c.startswith(p)
            for p in [
                "active_months",
                "amount_growth",
                "cp_growth",
                "growth_acceleration",
                "monthly_volatility",
                "counterparty_retention",
            ]
        )
    ],
    "Структура сети": [
        c
        for c in gbm_feature_cols
        if any(
            c.startswith(p)
            for p in [
                "unique_",
                "payee_ratio",
                "payer_ratio",
                "cp_per_month",
                "herfindahl_index",
                "amt_per_cp",
            ]
        )
    ],
}

ablation_results = []
full_auc = roc_auc_score(y_val, clf.predict_proba(X_val)[:, 1])
ablation_results.append(
    {
        "Группа": "Все признаки (baseline)",
        "N_features": len(gbm_feature_cols),
        "AUC-ROC": round(full_auc, 4),
        "Delta AUC": 0.0,
    }
)

for group_name, group_cols in FEATURE_GROUPS.items():
    present = [c for c in group_cols if c in gbm_feature_cols]
    if not present:
        continue
    remaining = [c for c in gbm_feature_cols if c not in present]
    if len(remaining) < 3:
        continue
    remaining_idx = [gbm_feature_cols.index(c) for c in remaining]
    X_tr_abl = X_tr[:, remaining_idx]
    X_val_abl = X_val[:, remaining_idx]
    try:
        import lightgbm as lgb

        abl_clf = lgb.LGBMClassifier(
            n_estimators=300,
            learning_rate=0.05,
            num_leaves=63,
            min_child_samples=20,
            subsample=0.8,
            colsample_bytree=0.8,
            n_jobs=-1,
            random_state=42,
            verbose=-1,
        )
        abl_clf.fit(
            X_tr_abl,
            y_tr,
            eval_set=[(X_val_abl, y_val)],
            callbacks=[lgb.early_stopping(20, verbose=False), lgb.log_evaluation(period=-1)],
        )
    except ImportError:
        abl_clf = GradientBoostingClassifier(
            n_estimators=200, max_depth=4, learning_rate=0.05, random_state=42
        )
        abl_clf.fit(X_tr_abl, y_tr)
    abl_auc = roc_auc_score(y_val, abl_clf.predict_proba(X_val_abl)[:, 1])
    ablation_results.append(
        {
            "Группа": f"Без '{group_name}' ({len(present)})",
            "N_features": len(remaining),
            "AUC-ROC": round(abl_auc, 4),
            "Delta AUC": round(abl_auc - full_auc, 4),
        }
    )

abl_df = pd.DataFrame(ablation_results)
print("Ablation Study — влияние групп признаков:")
display(abl_df)

if len(abl_df) > 1:
    fig, ax = plt.subplots(figsize=(10, 5))
    colors = ["#2ca02c"] + ["#d62728" if d < -0.01 else "#1f77b4" for d in abl_df["Delta AUC"][1:]]
    ax.barh(abl_df["Группа"], abl_df["AUC-ROC"], color=colors, edgecolor="black", alpha=0.8)
    ax.axvline(full_auc, color="green", linestyle="--", label=f"Baseline={full_auc:.3f}")
    ax.set_xlabel("AUC-ROC")
    ax.set_title("Ablation: AUC-ROC при удалении групп признаков")
    ax.legend()
    for i, (_, row) in enumerate(abl_df.iterrows()):
        ax.text(row["AUC-ROC"] + 0.002, i, f"{row['Delta AUC']:+.4f}", va="center", fontsize=9)
    plt.tight_layout()
    plt.show()

---
## 3.5. Важность признаков (Feature Importance)

Какие признаки GBM использует для разделения «лучших клиентов» от остальных?
Это говорит нам: **что именно делает клиента похожим на эталон**.

---
## 4. Ранжирование проспектов

In [ ]:
# ruff: noqa: F821
prospects_mask = full_df["clientcounterparty_flag"].isin(["N", "?"])
prospects = full_df[prospects_mask].sort_values("lookalike_score", ascending=False)

print(f"Total prospects: {len(prospects):,}")
print(
    f"Prospects with score >= {SCORE_THRESHOLD}: "
    f"{(prospects['lookalike_score'] >= SCORE_THRESHOLD).sum():,}"
)

display_cols = [
    "client_name",
    "client_type_name",
    "clientcounterparty_flag",
    "sparkokved_ccode",
    "addrref_city_name",
    "srvpackagesegment_ccode",
    "behavioral_segment",
    "total_amount",
    "unique_counterparties",
    "direction_ratio",
    "lookalike_score",
]
display_cols = [c for c in display_cols if c in prospects.columns]

print(f"\nTop-{TOP_N_PROSPECTS} prospects by lookalike_score:")
display(prospects[display_cols].head(TOP_N_PROSPECTS))

In [ ]:
# ruff: noqa: F821
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for label, mask in [("Clients (Y)", clients_mask), ("Prospects (N/?)", prospects_mask)]:
    subset = full_df.loc[mask, "lookalike_score"].dropna()
    axes[0].hist(subset, bins=50, alpha=0.5, label=f"{label} ({len(subset):,})", edgecolor="black")
axes[0].axvline(SCORE_THRESHOLD, color="red", linestyle="--", label=f"Threshold {SCORE_THRESHOLD}")
axes[0].set_xlabel("Lookalike Score")
axes[0].set_ylabel("Count")
axes[0].set_title("Score Distribution: Clients vs Prospects")
axes[0].legend()

top_prospects = prospects.head(TOP_N_PROSPECTS)
if "behavioral_segment" in top_prospects.columns:
    seg_counts = top_prospects["behavioral_segment"].value_counts().sort_index()
    axes[1].bar(seg_counts.index.astype(str), seg_counts.values, edgecolor="black", alpha=0.7)
    axes[1].set_xlabel("Behavioral Segment")
    axes[1].set_ylabel("Count")
    axes[1].set_title(f"Top-{TOP_N_PROSPECTS} Prospects by Segment")

plt.tight_layout()
plt.show()

---
## 5. Анализ: профиль top-проспектов vs эталон

In [ ]:
# ruff: noqa: F821
compare_cols = [
    "total_amount",
    "tx_count",
    "unique_counterparties",
    "direction_ratio",
    "active_months",
    "amount_growth",
]
for c in ["tx_per_month", "cp_per_month", "payee_ratio", "avg_balance", "product_count"]:
    if c in full_df.columns:
        compare_cols.append(c)
compare_cols = [c for c in compare_cols if c in full_df.columns]

# compare: index=compare_cols (фичи), columns=группы
compare = pd.DataFrame(
    {
        "Reference (best clients)": full_df.loc[reference_mask, compare_cols].mean(),
        f"Top-{TOP_N_PROSPECTS} prospects": top_prospects[compare_cols].mean(),
        "All prospects": prospects[compare_cols].mean(),
    }
)

# Убираем строки где все значения NaN (нет данных в источнике — напр. product_count)
# Это предотвращает RuntimeWarning: All-NaN slice encountered в MinMaxScaler
compare = compare.loc[~compare.isnull().all(axis=1)]

print("Profile comparison:")
display(compare.round(2))

norm_compare = MinMaxScaler().fit_transform(compare.T)
norm_compare_df = pd.DataFrame(norm_compare, index=compare.columns, columns=compare.index)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

bar_cols = [c for c in compare.index[:6]]
x = np.arange(len(bar_cols))
width = 0.28
ref_vals = compare.loc[bar_cols, "Reference (best clients)"]
top_vals = compare.loc[bar_cols, f"Top-{TOP_N_PROSPECTS} prospects"]
all_vals = compare.loc[bar_cols, "All prospects"]

ref_norm = ref_vals / ref_vals.replace(0, np.nan)
top_norm = top_vals / ref_vals.replace(0, np.nan)
all_norm = all_vals / ref_vals.replace(0, np.nan)

axes[0].bar(
    x - width, ref_norm, width, label="Reference", color="#2ca02c", alpha=0.85, edgecolor="black"
)
axes[0].bar(
    x,
    top_norm,
    width,
    label=f"Top-{TOP_N_PROSPECTS}",
    color="#1f77b4",
    alpha=0.85,
    edgecolor="black",
)
axes[0].bar(
    x + width,
    all_norm,
    width,
    label="All prospects",
    color="#ff7f0e",
    alpha=0.85,
    edgecolor="black",
)
axes[0].axhline(1.0, color="green", linestyle="--", alpha=0.5, label="Эталон = 1.0")
axes[0].set_xticks(x)
axes[0].set_xticklabels([c[:12] for c in bar_cols], rotation=30, ha="right")
axes[0].set_ylabel("Относительно эталона")
axes[0].set_title("Профиль: топ-проспекты vs эталон (нормировано)")
axes[0].legend(fontsize=8)

radar_cols = list(compare.index[: min(7, len(compare.index))])
angles = np.linspace(0, 2 * np.pi, len(radar_cols), endpoint=False).tolist()
angles += angles[:1]
ax_r = axes[1]
ax_r.remove()
ax_r = fig.add_subplot(1, 2, 2, polar=True)

for i, (group, row) in enumerate(norm_compare_df[radar_cols].iterrows()):
    vals = row.tolist() + [row.tolist()[0]]
    color = ["#2ca02c", "#1f77b4", "#ff7f0e"][i % 3]
    ax_r.plot(angles, vals, "o-", label=group, color=color, linewidth=2)
    ax_r.fill(angles, vals, alpha=0.07, color=color)
ax_r.set_xticks(angles[:-1])
ax_r.set_xticklabels([c[:10] for c in radar_cols], fontsize=8)
ax_r.set_title("Радар-профиль", fontsize=11, pad=15)
ax_r.legend(loc="upper right", bbox_to_anchor=(1.4, 1.1), fontsize=8)

plt.suptitle("Профиль top-проспектов vs эталон vs все проспекты", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

print("\n📌 ИНТЕРПРЕТАЦИЯ ПРОФИЛЯ:")
print("─" * 60)
for col in compare.index:
    ref_v = compare.loc[col, "Reference (best clients)"]
    top_v = compare.loc[col, f"Top-{TOP_N_PROSPECTS} prospects"]
    if pd.isna(ref_v) or ref_v == 0:
        continue
    ratio = top_v / ref_v
    trend = "✅" if ratio >= 0.7 else ("⚠️" if ratio >= 0.4 else "❌")
    print(
        f"  {trend} {col:28s}: top-{TOP_N_PROSPECTS} = {top_v:8.2f}  "
        f"(эталон = {ref_v:.2f}, покрытие {ratio:.0%})"
    )

---
## 6. Lift-анализ

Насколько модель лучше случайного выбора?

In [ ]:
# ruff: noqa: F821
if "clientchange_date" in full_df.columns and acquired_mask.sum() > 0:
    ranked = full_df.sort_values("lookalike_score", ascending=False).copy()
    ranked["rank_pct"] = np.arange(1, len(ranked) + 1) / len(ranked)
    ranked["is_acquired"] = ranked.index.isin(full_df[acquired_mask].index)

    pct_points = [0.01, 0.02, 0.05, 0.10, 0.20, 0.30, 0.50]
    lifts = []
    for pct in pct_points:
        top_k = ranked[ranked["rank_pct"] <= pct]
        acquired_in_top = int(top_k["is_acquired"].sum())
        expected = acquired_mask.sum() * pct
        lift = acquired_in_top / max(expected, 1)
        lifts.append(
            {
                "top_%": f"{pct * 100:.0f}%",
                "acquired_found": acquired_in_top,
                "expected_random": round(expected),
                "lift": round(lift, 2),
                "lift_str": f"{lift:.2f}x",
            }
        )

    lift_df = pd.DataFrame(lifts)
    print("Lift analysis (acquired clients in top-K% by score):")
    display(lift_df[["top_%", "acquired_found", "expected_random", "lift_str"]])

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    xs = [float(row["top_%"].strip("%")) for row in lifts]
    lift_vals = [row["lift"] for row in lifts]

    axes[0].plot(xs, lift_vals, "o-", linewidth=2, color="#1f77b4", markersize=8)
    axes[0].axhline(1.0, color="gray", linestyle="--", alpha=0.7, label="Random baseline")
    for x_pt, y_pt in zip(xs, lift_vals, strict=False):
        axes[0].annotate(
            f"{y_pt:.2f}x",
            (x_pt, y_pt),
            textcoords="offset points",
            xytext=(0, 8),
            ha="center",
            fontsize=9,
        )
    axes[0].set_xlabel("Top % by score")
    axes[0].set_ylabel("Lift")
    axes[0].set_title("Lift Curve: модель vs случайный выбор")
    axes[0].legend()
    axes[0].set_ylim(0.8, max(lift_vals) * 1.2)

    found_vals = [row["acquired_found"] for row in lifts]
    total_acquired = acquired_mask.sum()
    gain_pct = [v / total_acquired * 100 for v in found_vals]

    axes[1].plot(xs, gain_pct, "o-", linewidth=2, color="#2ca02c", label="Модель", markersize=8)
    axes[1].plot([0] + xs, [0] + xs, "--", color="gray", label="Random baseline")
    axes[1].fill_between(xs, gain_pct, xs[: len(xs)], alpha=0.15, color="#2ca02c")
    axes[1].set_xlabel("Top % отобрано")
    axes[1].set_ylabel("% привлечённых клиентов найдено")
    axes[1].set_title("Cumulative Gain Curve")
    axes[1].legend()

    plt.suptitle("Качество ранжирования модели", fontsize=13)
    plt.tight_layout()
    plt.show()

    top5_lift = next(row["lift"] for row in lifts if row["top_%"] == "5%")
    top10_lift = next(row["lift"] for row in lifts if row["top_%"] == "10%")
    top5_gain = next(row["acquired_found"] for row in lifts if row["top_%"] == "5%")

    print("\n📊 ИНТЕРПРЕТАЦИЯ LIFT-АНАЛИЗА:")
    print("─" * 60)
    print(f"  • Модель в топ-5% находит в {top5_lift:.2f}x больше привлечённых,")
    print(
        f"    чем случайный выбор ({top5_gain:,} vs {int(acquired_mask.sum() * 0.05):,} ожидаемых)"
    )
    print(f"  • Lift {top10_lift:.2f}x в топ-10% — модель эффективно отделяет")
    print("    «конвертируемых» контрагентов от остальных")
    if top5_lift > 1.5:
        print("  • ✅ Хороший lift: модель значительно лучше случайного выбора")
    elif top5_lift > 1.2:
        print("  • ⚠️  Умеренный lift: работает лучше random, но есть куда расти")
    else:
        print("  • ❌  Низкий lift: модель почти не лучше случайного выбора")
else:
    print("clientchange_date not available — lift analysis skipped.")

---
## 6.5. Сравнение методов и технологий

Оцениваем все 4 метода скоринга по единой метрике — **separation index** и **lift** на привлечённых клиентах. Технологическое сравнение — в итоговой таблице.

In [ ]:
# ruff: noqa: F821
# ── Сравнение методов: Centroid vs kNN vs GBM vs Ensemble ────────────────────
# Separation index = mean(score | reference) - mean(score | non-reference)
# Показывает насколько метод разделяет эталонных клиентов от остальных.
# Lift (если доступен clientchange_date): во сколько раз метод лучше random
# при отборе топ-K% базы.

METHODS = {
    "Centroid (baseline)": "score_centroid",
    f"kNN (k={K_NEIGHBORS})": "score_knn",
    f"GBM ({clf_name.split()[0]})": "score_gbm_norm",
    "Ensemble (рек.)": "lookalike_score",
}

# Boolean arrays aligned with full_df rows
is_ref_arr = reference_mask.values
has_lift_data = "clientchange_date" in full_df.columns and acquired_mask.sum() > 0
if has_lift_data:
    # Index.isin() returns a numpy array directly — no .values needed
    is_acq_arr = full_df.index.isin(full_df[acquired_mask].index)
else:
    is_acq_arr = None

method_scores_vals = {}
for mname, col in METHODS.items():
    if col in full_df.columns:
        method_scores_vals[mname] = full_df[col].fillna(0).values

# ── Таблица метрик ────────────────────────────────────────────────────────────
rows = []
for mname, scores in method_scores_vals.items():
    sep = float(scores[is_ref_arr].mean() - scores[~is_ref_arr].mean())
    row: dict = {"Метод": mname, "Separation↑": round(sep, 4)}

    if has_lift_data:
        ranked_idx = np.argsort(scores)[::-1]
        for pct in [0.05, 0.10, 0.20]:
            k_top = max(1, int(len(scores) * pct))
            top_mask = np.zeros(len(scores), dtype=bool)
            top_mask[ranked_idx[:k_top]] = True
            acq_in_top = int(is_acq_arr[top_mask].sum())
            expected = is_acq_arr.sum() * pct
            row[f"Lift@{int(pct * 100)}%"] = f"{acq_in_top / max(expected, 1):.2f}x"

    rows.append(row)

comparison_df = pd.DataFrame(rows).set_index("Метод")
print("📊 Сравнение методов lookalike-скоринга:")
display(comparison_df)

# ── Графики ───────────────────────────────────────────────────────────────────
_colors = ["#aec7e8", "#ffbb78", "#98df8a", "#d62728"]
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Левый: Separation index
sep_vals = {
    mname: float(scores[is_ref_arr].mean() - scores[~is_ref_arr].mean())
    for mname, scores in method_scores_vals.items()
}
bars = axes[0].bar(
    list(sep_vals.keys()),
    list(sep_vals.values()),
    color=_colors[: len(sep_vals)],
    edgecolor="black",
    alpha=0.85,
)
axes[0].set_ylabel("Separation Index\n(mean_ref − mean_non_ref)")
axes[0].set_title("Разделяющая способность методов (выше = лучше)", fontsize=11)
axes[0].set_xticklabels(list(sep_vals.keys()), rotation=15, ha="right")
for bar, val in zip(bars, sep_vals.values(), strict=False):
    axes[0].text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.002,
        f"{val:.3f}",
        ha="center",
        va="bottom",
        fontsize=9,
    )

# Правый: Lift curves (если доступны) или Score distribution для ref vs non-ref
if has_lift_data:
    pct_pts = [0.01, 0.02, 0.05, 0.10, 0.20, 0.30]
    for i, (mname, scores) in enumerate(method_scores_vals.items()):
        ranked_idx = np.argsort(scores)[::-1]
        lc = []
        for pct in pct_pts:
            k_top = max(1, int(len(scores) * pct))
            top_mask = np.zeros(len(scores), dtype=bool)
            top_mask[ranked_idx[:k_top]] = True
            expected = is_acq_arr.sum() * pct
            lc.append(int(is_acq_arr[top_mask].sum()) / max(expected, 1))
        xs = [p * 100 for p in pct_pts]
        style = ["--", ":", "-.", "-"][i % 4]
        axes[1].plot(
            xs,
            lc,
            f"o{style}",
            linewidth=2,
            color=_colors[i % len(_colors)],
            label=mname,
            markersize=7,
        )
    axes[1].axhline(1.0, color="gray", linestyle="--", alpha=0.5, label="Random")
    axes[1].set_xlabel("Top % by score")
    axes[1].set_ylabel("Lift")
    axes[1].set_title("Lift Curves по методам", fontsize=11)
    axes[1].legend(fontsize=8)
else:
    for i, (mname, scores) in enumerate(method_scores_vals.items()):
        axes[1].hist(
            scores[is_ref_arr],
            bins=30,
            alpha=0.45,
            label=f"{mname} (ref)",
            color=_colors[i % len(_colors)],
        )
    axes[1].set_xlabel("Score")
    axes[1].set_title("Score: эталонные клиенты (распределение)", fontsize=11)
    axes[1].legend(fontsize=8)

plt.suptitle("Сравнение методов: Centroid vs kNN vs GBM vs Ensemble", fontsize=13)
plt.tight_layout()
plt.show()

# ── Таблица технологий ────────────────────────────────────────────────────────
print("\n" + "═" * 90)
print("🏆  СРАВНЕНИЕ ТЕХНОЛОГИЙ LOOKALIKE-СКОРИНГА")
print("═" * 90)
_hdr = ("Метод", "Сложн.", "Скор.", "Интерпрет.", "Leakage-safe", "Production")
_sep = ("─" * 22, "─" * 6, "─" * 6, "─" * 14, "─" * 13, "─" * 18)
_rows = [
    ("Centroid baseline", "★☆☆", "⚡⚡⚡", "✅ высокая", "✅ всегда", "✅ нижняя планка"),
    (f"kNN cosine k={K_NEIGHBORS}", "★★☆", "⚡⚡", "✅ понятная", "✅ всегда", "✅ надёжный"),
    ("LightGBM (whitelist)", "★★★", "⚡⚡", "⚠️ нужен SHAP", "⚠️ whitelist", "✅ лучший F1"),
    ("LightGBM + SHAP", "★★★", "⚡", "✅ dir+mag", "⚠️ whitelist", "✅ лучший ROC-AUC"),
    ("Ensemble (рекоменд.)", "★★★", "⚡⚡", "⚠️ средняя", "⚠️ whitelist", "✅ лучший lift"),
]
for row in [_hdr, _sep, *_rows]:
    print(f"  {row[0]:23s}  {row[1]:7s}  {row[2]:7s}  {row[3]:15s}  {row[4]:14s}  {row[5]}")

print()
best_sep_name = max(sep_vals, key=sep_vals.get)
print("📝 ВЫВОДЫ:")
print("─" * 60)
print(f"  • Лучшая separation: {best_sep_name}  (sep={sep_vals[best_sep_name]:.3f})")
print("  • Centroid — простейший baseline, нижняя планка качества")
print("  • kNN — учитывает форму облака, устойчив к выбросам в эталоне")
print(f"  • {clf_name.split()[0]} — нелинейные паттерны; whitelist защищает от target leakage")
print("  • SHAP даёт направление влияния признака: ↑/↓, а не просто величину")
print("  • Ensemble стабилизирует: если один метод ошибается, второй корректирует")

---
## 7. Скоровые зоны (Score Bands)

Разбиение проспектов на бизнес-группы по уровню score для таргетирования.

In [ ]:
# ── Score Bands ──────────────────────────────────────────────────────────────
prospects_mask = full_df["clientcounterparty_flag"].isin(["N", "?"])
prospects = full_df[prospects_mask].sort_values("lookalike_score", ascending=False)

band_labels = []
for _, row in prospects.iterrows():
    score = row["lookalike_score"]
    assigned = "Cold"
    for band_name, (lo, hi) in SCORE_BANDS.items():
        if lo <= score < hi or (band_name == "Hot" and score >= lo):
            assigned = band_name
            break
    band_labels.append(assigned)

full_df.loc[prospects_mask, "score_band"] = band_labels

band_order = ["Hot", "Warm", "Medium", "Cool", "Cold"]
band_colors = {
    "Hot": "#d62728",
    "Warm": "#ff7f0e",
    "Medium": "#ffdd57",
    "Cool": "#1f77b4",
    "Cold": "#aec7e8",
}

band_stats = []
for band_name in band_order:
    band_mask = full_df["score_band"] == band_name
    n = int(band_mask.sum())
    if n == 0:
        continue
    band_data = full_df[band_mask]
    pct_etalon = (
        (band_data.index.isin(full_df[reference_mask].index)).mean()
        if reference_mask.sum() > 0
        else np.nan
    )
    band_stats.append(
        {
            "Band": band_name,
            "Count": n,
            "% of total": round(n / len(prospects) * 100, 1),
            "Mean Score": round(band_data["lookalike_score"].mean(), 3),
            "P(etalon)": round(pct_etalon, 3),
            "Mean Amount": round(band_data["total_amount"].mean(), 0)
            if "total_amount" in band_data.columns
            else "N/A",
        }
    )

band_stats_df = pd.DataFrame(band_stats)
print("Скоровые зоны (Score Bands):")
display(band_stats_df)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
valid_bands = [b for b in band_order if (full_df["score_band"] == b).sum() > 0]
valid_counts = [(full_df["score_band"] == b).sum() for b in valid_bands]
axes[0].bar(
    valid_bands,
    valid_counts,
    color=[band_colors[b] for b in valid_bands],
    edgecolor="black",
    alpha=0.85,
)
axes[0].set_ylabel("Количество проспектов")
axes[0].set_title("Распределение по скоровым зонам")
for i, (_b, c) in enumerate(zip(valid_bands, valid_counts, strict=False)):
    axes[0].text(i, c + len(prospects) * 0.01, f"{c:,}", ha="center", fontsize=9)

for bn in band_order:
    bd = full_df[(full_df["score_band"] == bn) & prospects_mask]
    if len(bd) > 0:
        axes[1].hist(
            bd["lookalike_score"],
            bins=20,
            alpha=0.6,
            color=band_colors[bn],
            label=bn,
            edgecolor="black",
        )
axes[1].set_xlabel("Lookalike Score")
axes[1].set_ylabel("Count")
axes[1].set_title("Распределение score по зонам")
axes[1].legend()

plt.suptitle("Score Bands: бизнес-зоны для таргетирования", fontsize=13)
plt.tight_layout()
plt.show()

---
## 8. Рекомендации по кампаниям

In [ ]:
# ── Campaign Recommendations ─────────────────────────────────────────────────
_recs = {
    "Hot": "Персональный менеджер. Индивидуальные предложения. Приоритетная обработка.",
    "Warm": "Таргетированные звонки и email-кампании. Пакетные предложения.",
    "Medium": "Массовые кампании. Digital-каналы. Промо-предложения.",
    "Cool": "Мониторинг. Включить в nurturing-программу. Ожидать роста.",
    "Cold": "Не тратить ресурсы. Автоматический мониторинг раз в квартал.",
}

if "total_amount" in full_df.columns and reference_mask.sum() > 0:
    ref_avg_amount = full_df.loc[reference_mask, "total_amount"].mean()
    print(f"Средний оборот эталонного клиента: {ref_avg_amount:,.0f} руб.\n")

print("=" * 80)
print("РЕКОМЕНДАЦИИ ПО КАМПАНИЯМ")
print("=" * 80)

for band_name in ["Hot", "Warm", "Medium", "Cool", "Cold"]:
    band_data = full_df[full_df["score_band"] == band_name]
    n = len(band_data)
    if n == 0:
        continue
    print(f"\n{'─' * 70}")
    print(f"  {band_name} ({n:,} проспектов)")
    print(f"  {_recs[band_name]}")
    if "total_amount" in band_data.columns:
        mean_amt = band_data["total_amount"].mean()
        print(f"  Средний текущий оборот: {mean_amt:,.0f} руб.")

total_hot_warm = full_df[full_df["score_band"].isin(["Hot", "Warm"])].shape[0]
print(f"\n{'─' * 70}")
print(f"\nРекомендуемый размер кампании: {total_hot_warm:,} проспектов (Hot + Warm)")

---
## 9. Model Card

| Параметр | Значение |
|----------|----------|
| **Модель** | LightGBM / XGBoost / sklearn GBM (fallback) |
| **Задача** | Бинарная классификация: эталонный клиент vs остальные |
| **Эталон** | Многофакторный composite quality score (6 измерений), топ 20% |
| **Валидация** | Stratified 5-Fold CV |
| **Ensemble** | Stacking (LR) или Fixed-weight (0.65*GBM + 0.35*kNN) |
| **Leakage protection** | Whitelist поведенческих признаков |
| **Score bands** | Hot / Warm / Medium / Cool / Cold |

### Ограничения
- Модель обучена на данных за 1 год (Jan-Dec 2025)
- Контрагенты без транзакций не попадают в выборку
- Рекомендуется переобучение каждые 6 месяцев

### Bias disclaimer
- Возможно смещение к крупным клиентам с высокой активностью
- Рекомендуется мониторинг распределения score по ОКВЭД и регионам

In [ ]:
# ruff: noqa: F821
import json as _json

scores_path = os.path.join(OUTPUT_DIR, "lookalike_scores.parquet")
full_df.to_parquet(scores_path)
print(f"Full scores saved: {scores_path}")

model_path = os.path.join(OUTPUT_DIR, "lookalike_gbm.pkl")
with open(model_path, "wb") as f:
    pickle.dump({"model": clf, "model_name": clf_name, "feature_names": feature_names}, f)
print(f"GBM model saved: {model_path}")

# Model card JSON
model_card = {
    "model_name": clf_name,
    "task": "binary_classification",
    "target": "reference_client",
    "reference_selection": "composite_quality_score_top20pct",
    "cv_folds": CV_FOLDS,
    "cv_auc_mean": round(float(cv_df["AUC-ROC"].mean()), 4),
    "cv_auc_std": round(float(cv_df["AUC-ROC"].std()), 4),
    "cv_separation_mean": round(float(cv_df["Separation"].mean()), 4),
    "brier_score": round(float(brier), 4),
    "ece": round(float(ece), 4),
    "ensemble_method": ENSEMBLE_METHOD if "ENSEMBLE_METHOD" in dir() else "Fixed-weight",
    "optimal_threshold": round(float(OPTIMAL_THRESHOLD), 4),
    "period": "2025-01-01 to 2025-12-31",
    "n_features": len(gbm_feature_cols),
    "n_reference": int(reference_mask.sum()),
    "n_prospects": int(prospects_mask.sum()),
    "score_bands": {b: f"{lo:.1f}-{hi:.1f}" for b, (lo, hi) in SCORE_BANDS.items()},
}
card_path = os.path.join(OUTPUT_DIR, "model_card.json")
with open(card_path, "w", encoding="utf-8") as f:
    _json.dump(model_card, f, ensure_ascii=False, indent=2)
print(f"Model card saved: {card_path}")

export_cols = [
    c
    for c in [
        "client_name",
        "client_type_name",
        "clientcounterparty_flag",
        "sparkokved_ccode",
        "addrref_city_name",
        "reg_city_name",
        "srvpackagesegment_ccode",
        "behavioral_segment",
        "segment_label",
        "total_amount",
        "tx_count",
        "unique_counterparties",
        "direction_ratio",
        "active_months",
        "amount_growth",
        "score_centroid",
        "score_knn",
        "score_gbm",
        "score_gbm_norm",
        "score_stacking",
        "lookalike_score",
        "score_band",
    ]
    if c in prospects.columns
]

top_path = os.path.join(OUTPUT_DIR, "top_prospects.parquet")
prospects[export_cols].head(TOP_N_PROSPECTS * 5).to_parquet(top_path)
print(f"Top prospects saved: {top_path}")

try:
    xlsx_path = os.path.join(OUTPUT_DIR, "top_prospects.xlsx")
    (
        prospects[export_cols]
        .head(TOP_N_PROSPECTS * 5)
        .rename(
            columns={
                "lookalike_score": "score_final",
                "sparkokved_ccode": "okved",
                "addrref_city_name": "city",
                "srvpackagesegment_ccode": "model_segment",
            }
        )
        .to_excel(xlsx_path, index=True)
    )
    print(f"Excel saved: {xlsx_path}")
except Exception as e:
    print(f"Excel export skipped ({e})")

print(f"\n{'=' * 60}")
print("ИТОГО:")
print(f"  Модель: {clf_name}")
print(f"  CV AUC-ROC: {cv_df['AUC-ROC'].mean():.3f} +/- {cv_df['AUC-ROC'].std():.3f}")
print(f"  Brier Score: {brier:.4f}, ECE: {ece:.4f}")
print(f"  Эталонная группа: {reference_mask.sum():,}")
print(f"  Проспекты: {prospects_mask.sum():,}")
if "score_band" in full_df.columns:
    for band in ["Hot", "Warm", "Medium", "Cool", "Cold"]:
        n = (full_df["score_band"] == band).sum()
        if n > 0:
            print(f"  {band:8s}: {n:>6,}")

---

## Глоссарий

| Термин | Описание |
|--------|----------|
| **Эталонная группа** | Лучшие привлечённые клиенты: composite quality score, топ 20% |
| **Centroid baseline** | Cosine similarity с центроидом эталонов. Нижняя планка |
| **kNN cosine** | Среднее cosine similarity к K ближайшим эталонам |
| **LightGBM** | Гистограммный GBM. 5-10x быстрее sklearn |
| **GBM whitelist** | Только поведенческие признаки. Защита от target leakage |
| **SHAP** | SHapley Additive exPlanations. Направление + вклад признака |
| **Ensemble** | Stacking LR или 0.65*GBM + 0.35*kNN |
| **Separation Index** | mean(score ref) - mean(score non-ref) |
| **Lift** | Во сколько раз метод лучше random при отборе топ-K% |
| **Stratified K-Fold CV** | K-fold с сохранением пропорции классов |
| **AUC-ROC** | Area Under ROC Curve. 0.5=random, 1.0=perfect |
| **PR-AUC** | Area Under Precision-Recall Curve |
| **Brier Score** | MSE вероятностей. Ниже = лучше |
| **ECE** | Expected Calibration Error. Ниже = лучше |
| **Youden's J** | TPR - FPR. Оптимальный порог для ROC |
| **Stacking** | Meta-learner (LR) на OOF-предсказаниях базовых моделей |
| **Ablation Study** | Удаление группы признаков -> оценка влияния на AUC |
| **Score Bands** | Hot (0.8+), Warm (0.6-0.8), Medium (0.4-0.6), Cool (0.2-0.4), Cold (0-0.2) |
| **Target leakage** | Признак, закодирующий метку |

---

**Pipeline завершён.** Файлы в `data/`:
- `lookalike_scores.parquet` — все клиенты со scores
- `top_prospects.parquet` / `.xlsx` — топ-500 проспектов
- `lookalike_gbm.pkl` — обученная GBM модель
- `model_card.json` — метаданные модели для мониторинга